In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain_ollama import ChatOllama

import os 
from langchain_openai import AzureChatOpenAI
from langchain_ollama import OllamaLLM

model = OllamaLLM(
    model="gemma4:31b-cloud",
    temperature=0.7,
    num_predict=256,
)


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\nError generating summary: 'str' object has no attribute 'text'", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='8789bfdd-0936-480d-be02-5f6f7b3fdd0e'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='c36b91e5-e5b2-46b3-9b0f-f4978cd0f7af'),
              HumanMessage(content='*(Assuming the role of the newly inaugurated President of Lunapolis, standing at the podium in the Lunar Dome, addressing the Cheese Miners\' Union)*\n\n***\n\n**"Citizens of Lunapolis, and most importantly, the brave members of the Miners\' Union,**\n\nI stand before you today not just as your President, but as someone who knows that the very foundation of our lunar colony—the gold-hued, savory crust we call home—is carved out by your hands. \n\nFor too long, the administration

In [5]:
print(response["messages"][-1].content)

*(Assuming the role of the newly inaugurated President of Lunapolis, standing at the podium in the Lunar Dome, addressing the Cheese Miners' Union)*

***

**"Citizens of Lunapolis, and most importantly, the brave members of the Miners' Union,**

I stand before you today not just as your President, but as someone who knows that the very foundation of our lunar colony—the gold-hued, savory crust we call home—is carved out by your hands. 

For too long, the administration has treated the Great Gorgonzola Veins as a mere resource to be extracted, while treating the miners as mere tools of extraction. That ends today. I have read your manifesto, and I hear your demands.

To the Union leadership, I propose the **'Lunar Lactose Accord,'** based on three non-negotiable pillars:

**First: Safety and Sustainability.** 
We cannot ignore the 'Rind-Collapse' incidents of the last quarter. I am ordering an immediate audit of all structural supports in the Brie-Basins. No miner will be forced into a 

## Trim/delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='35ab3666-e99c-4d06-9cac-4d75dd70a78c'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='de52aa5b-c2ea-4d8d-bdc3-17167415df83', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='75d6eaae-7ad3-416e-ae91-3d0ef899f429'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='5f7a4d9c-fa32-453c-a2c6-9d5b301feb28', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='1e1e8772-4db0-4195-b7d1-ec41bd4cfa7c'),
              HumanMessage(content="I cannot determine the temperature of your device remotely.

In [11]:
print(response["messages"][-1].content)

I cannot determine the temperature of your device remotely. To check it, you can carefully feel the exterior casing or the power adapter:

*   **Does it feel excessively hot to the touch?** (This could indicate overheating).
*   **Does it feel unusually cold?** (This could indicate a power failure or that it hasn't been running).
*   **Does it feel room temperature?** (This suggests it isn't receiving power or attempting to boot).

**If the device is very hot**, unplug it and let it cool down for 30 minutes before trying again.

Returning to the troubleshooting: Since you mentioned it is plugged in and turned on, but you haven't confirmed if there are any **lights or indicators** (like a power LED, blinking light, or screen flicker), could you let me know if you see any?
